# 01 — Data Loading, Cleaning & Master Dataset Build

Purpose:
- Load raw Eurostat TSV files from `data/raw/eurostat/`
- Clean each KPI into country-year format
- Merge all KPI tables into one EU27 master panel
- Add government expenditure / COFOG spending layer
- Save final processed datasets into `data/processed/`

This notebook is intentionally only for data preparation. EDA is moved to `02_preliminary_eda.ipynb`.

In [1]:
# Core imports
from pathlib import Path
from functools import reduce
import sys

import pandas as pd

# Make src/ importable from this notebook
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.config import RAW_DIR, CLEANED_KPI_DIR, PROCESSED_DIR
from src.clean_eurostat import clean_eurostat_tsv, clean_government_expenditure

print("Project root:", PROJECT_ROOT)
print("Raw data folder:", RAW_DIR)
print("Processed data folder:", PROCESSED_DIR)

Project root: c:\Users\gilad\Documents\Data Analytics course 02_26\capstone-eu-structural-tradeoffs
Raw data folder: C:\Users\gilad\Documents\Data Analytics course 02_26\capstone-eu-structural-tradeoffs\data\raw\eurostat
Processed data folder: C:\Users\gilad\Documents\Data Analytics course 02_26\capstone-eu-structural-tradeoffs\data\processed


## 1. File Map

Put the raw downloaded Eurostat TSV files in:

`data/raw/eurostat/`

Expected filenames are listed below. Adjust the dictionary if your local filenames differ.

In [2]:
KPI_FILES = {
    "gdp_growth": "gdp_growth.tsv",
    "inflation": "inflation.tsv",
    "unemployment": "unemployment.tsv",
    "gini": "gini.tsv",
    "education": "education.tsv",
    "rnd": "rnd.tsv",
    "ict_specialists": "ict_specialists.tsv",
    "renewables": "renewables.tsv",
    "emissions": "emissions.tsv",
    "debt": "debt.tsv",
}

GOV_FILE = "government_expenditure.tsv"

# Check which files are visible
for value_name, filename in KPI_FILES.items():
    print(f"{filename:30s}", "OK" if (RAW_DIR / filename).exists() else "MISSING")

print(f"{GOV_FILE:30s}", "OK" if (RAW_DIR / GOV_FILE).exists() else "MISSING")

gdp_growth.tsv                 OK
inflation.tsv                  OK
unemployment.tsv               OK
gini.tsv                       OK
education.tsv                  OK
rnd.tsv                        OK
ict_specialists.tsv            OK
renewables.tsv                 OK
emissions.tsv                  OK
debt.tsv                       OK
government_expenditure.tsv     OK


## 2. Clean Eurostat KPI Tables

In [3]:
cleaned_tables = {}

for value_name, filename in KPI_FILES.items():
    filepath = RAW_DIR / filename

    df = clean_eurostat_tsv(filepath, value_name)
    cleaned_tables[value_name] = df

    # Save cleaned single-KPI file for traceability
    df.to_csv(CLEANED_KPI_DIR / f"{value_name}.csv", index=False)

    print(f"{value_name:18s} shape={df.shape}, countries={df['country'].nunique()}, years={df['year'].min()}-{df['year'].max()}, missing={df[value_name].isna().sum()}")

gdp_growth         shape=(468, 3), countries=39, years=2014-2025, missing=9
inflation          shape=(492, 3), countries=41, years=2014-2025, missing=21
unemployment       shape=(456, 3), countries=38, years=2014-2025, missing=13
gini               shape=(480, 3), countries=40, years=2014-2025, missing=45
education          shape=(468, 3), countries=39, years=2014-2025, missing=20
rnd                shape=(495, 3), countries=45, years=2014-2024, missing=41
ict_specialists    shape=(444, 3), countries=37, years=2014-2025, missing=19
renewables         shape=(440, 3), countries=40, years=2014-2024, missing=22
emissions          shape=(341, 3), countries=31, years=2014-2024, missing=0
debt               shape=(360, 3), countries=30, years=2014-2025, missing=0


## 3. Merge KPI Tables Into Master Panel

In [4]:
# Merge all KPI tables by country-year
kpi_dfs = [cleaned_tables[name] for name in KPI_FILES.keys()]

eu_master = reduce(
    lambda left, right: pd.merge(left, right, on=["country", "year"], how="outer"),
    kpi_dfs,
)

eu_master = eu_master.sort_values(["country", "year"]).reset_index(drop=True)

eu_master = eu_master[
    [
        "country",
        "year",
        "gdp_growth",
        "inflation",
        "unemployment",
        "gini",
        "education",
        "rnd",
        "ict_specialists",
        "renewables",
        "emissions",
        "debt",
    ]
]

print("Shape:", eu_master.shape)
display(eu_master.head())
display(eu_master.isna().sum().sort_values(ascending=False))

Shape: (594, 12)


,country,year,gdp_growth,inflation,unemployment,gini,education,rnd,ict_specialists,renewables,emissions,debt
0,AL,2014,2.2,NaN,NaN,NaN,NaN,NaN,NaN,31.856,NaN,NaN
1,AL,2015,2.2,NaN,NaN,NaN,NaN,NaN,NaN,34.913,NaN,NaN
2,AL,2016,3.9,NaN,NaN,NaN,NaN,NaN,NaN,36.953,NaN,NaN
3,AL,2017,3.3,NaN,NaN,36.8,NaN,NaN,NaN,35.776,NaN,NaN
4,AL,2018,3.7,1.8,NaN,35.4,NaN,NaN,NaN,36.572,NaN,NaN


emissions          253
debt               234
renewables         176
ict_specialists    169
gini               159
unemployment       151
education          146
rnd                140
gdp_growth         135
inflation          123
country              0
year                 0
dtype: int64

## 4. Validate Coverage by Country

This shows which countries have reliable coverage before filtering to EU27.

In [5]:
coverage_by_country = (
    eu_master
    .assign(non_missing_kpis=eu_master.drop(columns=["country", "year"]).notna().sum(axis=1))
    .groupby("country")
    .agg(
        years=("year", "nunique"),
        avg_kpis_available=("non_missing_kpis", "mean"),
        min_kpis_available=("non_missing_kpis", "min"),
        max_kpis_available=("non_missing_kpis", "max"),
    )
    .sort_values("avg_kpis_available", ascending=False)
)

display(coverage_by_country)
print("Countries:", eu_master["country"].nunique())

,years,avg_kpis_available,min_kpis_available,max_kpis_available
country,,,,
AT,12,9.750000,7,10
BE,12,9.750000,7,10
EE,12,9.750000,7,10
BG,12,9.750000,7,10
CZ,12,9.750000,7,10
CY,12,9.750000,7,10
DE,12,9.750000,7,10
EL,12,9.750000,7,10
ES,12,9.750000,7,10


Countries: 50


## 5. Filter to EU27 Countries

In [6]:
eu27_countries = [
    "AT", "BE", "BG", "HR", "CY", "CZ", "DK", "EE", "FI", "FR",
    "DE", "EL", "HU", "IE", "IT", "LV", "LT", "LU", "MT", "NL",
    "PL", "PT", "RO", "SK", "SI", "ES", "SE",
]

eu_master_eu27 = eu_master[eu_master["country"].isin(eu27_countries)].copy()

eu_master_eu27 = eu_master_eu27.sort_values(["country", "year"]).reset_index(drop=True)

print("Shape:", eu_master_eu27.shape)
print("Countries:", eu_master_eu27["country"].nunique())
print("Years:", eu_master_eu27["year"].min(), "-", eu_master_eu27["year"].max())

display(eu_master_eu27.head())
display(eu_master_eu27.isna().sum().sort_values(ascending=False))
display((eu_master_eu27.isna().mean().mul(100).round(1).sort_values(ascending=False)))

Shape: (324, 12)
Countries: 27
Years: 2014 - 2025


,country,year,gdp_growth,inflation,unemployment,gini,education,rnd,ict_specialists,renewables,emissions,debt
0,AT,2014,0.8,1.5,6.0,27.6,79.7,3.11,3.6,33.550,9.0,85.2
1,AT,2015,1.3,0.8,6.1,27.2,80.4,3.07,4.0,33.497,9.2,85.6
2,AT,2016,2.1,1.0,6.5,27.2,80.4,3.13,4.2,33.370,9.2,83.4
3,AT,2017,2.3,2.2,5.9,27.9,80.7,3.07,4.4,33.136,9.4,79.1
4,AT,2018,2.5,2.1,5.2,26.8,81.1,3.11,4.5,33.784,9.0,74.6


renewables         27
emissions          27
rnd                27
country             0
inflation           0
gdp_growth          0
year                0
unemployment        0
education           0
gini                0
ict_specialists     0
debt                0
dtype: int64

renewables         8.3
emissions          8.3
rnd                8.3
country            0.0
inflation          0.0
gdp_growth         0.0
year               0.0
unemployment       0.0
education          0.0
gini               0.0
ict_specialists    0.0
debt               0.0
dtype: float64

## 6. Forward-Fill Known 2025 KPI Gaps

Some 2025 values are missing because Eurostat has not published all indicators yet.  
For slow-moving indicators (`rnd`, `renewables`, `emissions`), we use within-country forward-fill.

In [7]:
eu_master_original = eu_master_eu27.copy()
eu_master_filled = eu_master_eu27.copy()

cols_to_fill = ["rnd", "renewables", "emissions"]

eu_master_filled[cols_to_fill] = (
    eu_master_filled
    .sort_values(["country", "year"])
    .groupby("country")[cols_to_fill]
    .ffill()
)

print("Missing values after KPI forward-fill:")
display(eu_master_filled.isna().sum().sort_values(ascending=False))

Missing values after KPI forward-fill:


country            0
year               0
gdp_growth         0
inflation          0
unemployment       0
gini               0
education          0
rnd                0
ict_specialists    0
renewables         0
emissions          0
debt               0
dtype: int64

## 7. Clean Government Expenditure / COFOG Layer

This adds public investment priority indicators (% of GDP): defense, economic affairs, environment, health, education, social protection, etc.

In [8]:
gov_wide = clean_government_expenditure(RAW_DIR / GOV_FILE)

print("Government spending wide shape:", gov_wide.shape)
display(gov_wide.head())
display(gov_wide.isna().sum().sort_values(ascending=False))

Government spending wide shape: (331, 11)


,country,year,defense_spending,economic_affairs_spending,fuel_energy_spending,communication_spending,environment_spending,health_spending,education_spending,social_protection_spending,total_gov_expenditure
0,AT,2014,0.6,7.0,0.2,0.0,0.4,8.2,5.0,21.4,52.4
1,AT,2015,0.6,6.0,0.3,0.0,0.4,8.3,5.0,21.3,51.2
2,AT,2016,0.6,5.9,0.3,0.1,0.4,8.3,4.9,21.1,50.6
3,AT,2017,0.6,6.1,0.3,0.1,0.4,8.3,4.9,20.7,49.8
4,AT,2018,0.6,6.0,0.2,0.1,0.4,8.3,4.8,20.3,49.2


environment_spending          1
fuel_energy_spending          1
economic_affairs_spending     1
social_protection_spending    1
education_spending            1
total_gov_expenditure         1
health_spending               1
defense_spending              0
year                          0
country                       0
communication_spending        0
dtype: int64

## 8. Merge Spending Layer with Master Dataset

In [9]:
eu_master_plus = eu_master_filled.merge(
    gov_wide,
    on=["country", "year"],
    how="left",
)

print("Merged shape:", eu_master_plus.shape)
display(eu_master_plus.head())
display(eu_master_plus.isna().sum().sort_values(ascending=False))

Merged shape: (324, 21)


,country,year,gdp_growth,inflation,unemployment,gini,education,rnd,ict_specialists,renewables,...,debt,defense_spending,economic_affairs_spending,fuel_energy_spending,communication_spending,environment_spending,health_spending,education_spending,social_protection_spending,total_gov_expenditure
0,AT,2014,0.8,1.5,6.0,27.6,79.7,3.11,3.6,33.550,...,85.2,0.6,7.0,0.2,0.0,0.4,8.2,5.0,21.4,52.4
1,AT,2015,1.3,0.8,6.1,27.2,80.4,3.07,4.0,33.497,...,85.6,0.6,6.0,0.3,0.0,0.4,8.3,5.0,21.3,51.2
2,AT,2016,2.1,1.0,6.5,27.2,80.4,3.13,4.2,33.370,...,83.4,0.6,5.9,0.3,0.1,0.4,8.3,4.9,21.1,50.6
3,AT,2017,2.3,2.2,5.9,27.9,80.7,3.07,4.4,33.136,...,79.1,0.6,6.1,0.3,0.1,0.4,8.3,4.9,20.7,49.8
4,AT,2018,2.5,2.1,5.2,26.8,81.1,3.11,4.5,33.784,...,74.6,0.6,6.0,0.2,0.1,0.4,8.3,4.8,20.3,49.2


fuel_energy_spending          27
social_protection_spending    27
education_spending            27
total_gov_expenditure         27
health_spending               27
economic_affairs_spending     27
environment_spending          27
communication_spending        26
defense_spending              26
country                        0
inflation                      0
gdp_growth                     0
year                           0
unemployment                   0
debt                           0
renewables                     0
emissions                      0
education                      0
rnd                            0
gini                           0
ict_specialists                0
dtype: int64

## 9. Forward-Fill 2025 Government Spending Gaps

Government expenditure is also missing for many 2025 rows. Since these are slow-moving %GDP policy-priority proxies, we forward-fill within country.

In [10]:
spending_cols = [
    "defense_spending",
    "economic_affairs_spending",
    "fuel_energy_spending",
    "communication_spending",
    "environment_spending",
    "health_spending",
    "education_spending",
    "social_protection_spending",
    "total_gov_expenditure",
]

# Only keep columns that exist, so the notebook is robust if one raw category is missing
spending_cols = [col for col in spending_cols if col in eu_master_plus.columns]

eu_master_plus[spending_cols] = (
    eu_master_plus
    .sort_values(["country", "year"])
    .groupby("country")[spending_cols]
    .ffill()
)

print("Missing values after spending forward-fill:")
display(eu_master_plus.isna().sum().sort_values(ascending=False))

Missing values after spending forward-fill:


country                       0
year                          0
gdp_growth                    0
inflation                     0
unemployment                  0
gini                          0
education                     0
rnd                           0
ict_specialists               0
renewables                    0
emissions                     0
debt                          0
defense_spending              0
economic_affairs_spending     0
fuel_energy_spending          0
communication_spending        0
environment_spending          0
health_spending               0
education_spending            0
social_protection_spending    0
total_gov_expenditure         0
dtype: int64

## 10. Save Processed Datasets

In [11]:
# Main outputs for EDA and Tableau

eu_master_original.to_csv(PROCESSED_DIR / "eu_master_original_eu27.csv", index=False)
eu_master_filled.to_csv(PROCESSED_DIR / "eu_master_filled.csv", index=False)
gov_wide.to_csv(PROCESSED_DIR / "government_spending_wide.csv", index=False)
eu_master_plus.to_csv(PROCESSED_DIR / "eu_master_plus.csv", index=False)

# Optional Excel copy for quick inspection
try:
    eu_master_plus.to_excel(PROCESSED_DIR / "eu_master_plus.xlsx", index=False)
except Exception as e:
    print("Excel export skipped:", e)

print("Saved files:")
for file in sorted(PROCESSED_DIR.iterdir()):
    print("-", file.name)

Saved files:
- .gitkeep
- eu_master_filled.csv
- eu_master_original_eu27.csv
- eu_master_plus.csv
- eu_master_plus.xlsx
- government_spending_wide.csv
